# Train EGE-UNet on Colab, persist everything to Google Drive

This notebook trains EGE-UNet on a Colab GPU while writing **all outputs** (logs, checkpoints, TensorBoard events, visualization images, `metrics.csv`, `test_results.json`) to a folder on your Google Drive, so nothing is lost when the session dies.

**Key concept: the Drive `WORK_DIR` *is* the experiment.**
- Re-running this notebook with the **same** `WORK_DIR` automatically **resumes** from its `checkpoints/latest.pth`.
- To start a **new** experiment, just pick a new `WORK_DIR` (e.g. change `RUN_NAME`).

**Prerequisites (one-time):**
1. Your fork/copy of this repo pushed to GitHub (set `REPO_URL` below).
2. The dataset zip `data_isic1718.zip` (from the repo's `data/` folder or the download link in `README.md`) uploaded to your Drive (set `DATA_ZIP` below).

Then: **Runtime → Run all**. If the session disconnects mid-training, simply run the notebook again from the top — training continues where it left off.

In [ ]:
# ---------------- Parameters ----------------
REPO_URL   = 'https://github.com/<your-username>/EGE-UNet.git'   # your repo
DATASET    = 'isic17'                                            # 'isic17' | 'isic18'
HPA_MODE   = 'learnable'                                         # 'learnable' (original) | 'frozen_ones' | 'none'
SEED       = 42                                                  # use 42, 43, 44 ... for multi-seed runs
RUN_NAME   = f'egeunet_{DATASET}_{HPA_MODE}_s{SEED}'             # new name => new experiment
WORK_DIR   = f'/content/drive/MyDrive/EGE-UNet-results/{RUN_NAME}'
DATA_ZIP   = '/content/drive/MyDrive/datasets/data_isic1718.zip' # where you uploaded the zip
EXTRA_ARGS = ''                                                  # e.g. '--epochs 300 --num-workers 2'

# local (fast) copy of the dataset; the zip extracts to isic2017/ and isic2018/
DATA_DIR = '/content/data_isic1718/' + ('isic2017' if DATASET == 'isic17' else 'isic2018')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo on first run, pull updates on later runs
import os
if not os.path.isdir('/content/EGE-UNet'):
    !git clone {REPO_URL} /content/EGE-UNet
else:
    !git -C /content/EGE-UNet pull --ff-only
%cd /content/EGE-UNet

In [ ]:
# All requirements are preinstalled on Colab; this is a fast no-op kept for portability
!pip install -q -r requirements.txt

### Stage the dataset to the local disk

Drive is a network (FUSE) mount: opening each of the thousands of small image files is a network round-trip, which makes epochs **10–100× slower** if you read the dataset from Drive directly. Instead we copy **one** zip (full sequential bandwidth) and unzip it to the local SSD. This cell is skipped automatically if the data is already staged in this session.

In [ ]:
if not os.path.isdir(DATA_DIR):
    !cp "{DATA_ZIP}" /content/data_isic1718.zip
    !unzip -q /content/data_isic1718.zip -d /content/data_isic1718
    !rm /content/data_isic1718.zip
!ls {DATA_DIR}

### (Optional) Live TensorBoard

Started *before* training so the curves update live. It reads from Drive, so it can lag a little behind.

In [ ]:
%load_ext tensorboard
from tensorboard import notebook
notebook.start('--logdir "' + WORK_DIR + '/summary"')

### Train (or resume)

This is the cell to re-run after any disconnect. If a `latest.pth` already exists in `WORK_DIR/checkpoints/`, you will see a line like `resuming model from .../latest.pth. resume_epoch: N` and training continues from epoch N+1. Re-running an already-finished run is a harmless no-op.

Notes:
- GHPA ablation matrix: run this notebook once per `(HPA_MODE, SEED)` combination, e.g. `learnable / frozen_ones / none` × seeds `42, 43, 44`. Each combination gets its own `RUN_NAME` folder on Drive; analyse them afterwards with `colab_analysis.ipynb`.
- `--num-workers 2` (via `EXTRA_ARGS`) speeds up data loading, but makes the augmentation RNG stream differ slightly from the paper's single-process setup.
- To intentionally restart from scratch in the same folder, add `--no-resume` to `EXTRA_ARGS` (or pick a new `RUN_NAME`).

In [ ]:
!python train.py --dataset {DATASET} --data-path "{DATA_DIR}" --work-dir "{WORK_DIR}" --hpa-mode {HPA_MODE} --seed {SEED} {EXTRA_ARGS}

### Inspect the artifacts stored on Drive

In [ ]:
import json
import pandas as pd

!find "{WORK_DIR}" -type f | sort

csv_path = f'{WORK_DIR}/metrics.csv'
if os.path.exists(csv_path):
    display(pd.read_csv(csv_path).tail(10))

results_path = f'{WORK_DIR}/test_results.json'
if os.path.exists(results_path):
    print(json.dumps(json.load(open(results_path)), indent=2))